# Assignment 2: The "Smart Labeling Pipeline" Challenge

**Total Marks: 20**

Build a cost-effective, high-quality labeling pipeline using human annotation, programmatic rules, and LLMs.

This notebook implements an end-to-end smart labeling pipeline to:
1. Establish gold standard through human annotation and measure inter-annotator agreement (6 marks)
2. Label data programmatically using weak supervision (Snorkel) (6 marks)
3. Optimize labeling budget using active learning (5 marks)
4. Leverage LLMs for bulk labeling and detect hallucinations (e.g. noisy labels) (3 marks)

## Setup and Imports

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from snorkel.labeling import labeling_function, PandasLFApplier, LFAnalysis
from snorkel.labeling.model import LabelModel
from statsmodels.stats.inter_rater import fleiss_kappa
import google.generativeai as genai
import time
from pathlib import Path
import re
print("Successful")

Successful


/mnt/c/GNSK/SEM-4/Software Tools and Techniques/assignment-2/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/tmp/ipykernel_1168/2489187220.py:13: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


## Task 1: The Human as Annotator (6 Marks)

**Objective:** Establish a "Gold Standard" dataset and measure human consensus.

### Part 1.1: Parse Annotator CSV Files

After annotating the first 100 reviews, export annotations from three annotators (A, B, C) as CSV files.
Parse these CSV files into clean DataFrames for analysis.

In [13]:
def parse_annotator_csv(csv_path):
    """
    Parses annotator CSV file into a clean DataFrame.
    
    Args:
        csv_path (str): Path to annotator CSV file
        
    Returns:
        pd.DataFrame: DataFrame with columns ['review', 'label']
                     where label is one of: 'Positive', 'Negative', 'Neutral'
    
    Note:
        - Handles Label Studio CSV exports (with 'annotations' column containing JSON)
        - Also handles simple CSV formats with direct 'review' and 'label' columns
        - Standardizes label values to: 'Positive', 'Negative', 'Neutral'
    """
    # Load CSV file using pd.read_csv()
    df = pd.read_csv(csv_path)
    
    # Method 1: Handle Label Studio exports (JSON in 'annotations' column)
    if 'annotations' in df.columns or 'annotation' in df.columns:
        reviews = []
        labels = []
        
        # Find the data column (contains review text)
        data_col = None
        for col in ['data', 'text', 'review']:
            if col in df.columns:
                data_col = col
                break
        
        # Find the annotations column
        annot_col = 'annotations' if 'annotations' in df.columns else 'annotation'
        
        for idx, row in df.iterrows():
            try:
                # Extract review text
                if data_col:
                    if isinstance(row[data_col], str) and row[data_col].startswith('{'):
                        # Data is JSON string, need to parse it
                        data_json = json.loads(row[data_col])
                        review_text = data_json.get('review', data_json.get('text', ''))
                    else:
                        review_text = row[data_col]
                else:
                    # Fallback: look for any column with text-like content
                    review_text = str(row.iloc[0])
                
                # Extract label from annotations
                annotations = row[annot_col]
                if isinstance(annotations, str):
                    # Parse JSON annotations
                    annot_json = json.loads(annotations)
                    if isinstance(annot_json, list) and len(annot_json) > 0:
                        # Get first annotation
                        result = annot_json[0].get('result', [])
                        if len(result) > 0:
                            # Extract label from 'value' -> 'choices'
                            label = result[0].get('value', {}).get('choices', [''])[0]
                        else:
                            label = ''
                    else:
                        label = ''
                else:
                    label = ''
                
                if review_text and label:
                    reviews.append(review_text)
                    labels.append(label)
            except (json.JSONDecodeError, KeyError, IndexError) as e:
                # Skip rows with parsing errors
                print(f"⚠ Warning: Could not parse row {idx}: {e}")
                continue
        
        result_df = pd.DataFrame({'review': reviews, 'label': labels})
    
    # Method 2: Handle simple CSV format with direct columns
    else:
        # Try to identify the label column
        label_cols = ['label', 'choice', 'sentiment', 'annotation', 'Label', 'Choice']
        label_col = None
        for col in label_cols:
            if col in df.columns:
                label_col = col
                break
        
        # Try to identify the review column
        review_cols = ['review', 'text', 'Review', 'Text']
        review_col = None
        for col in review_cols:
            if col in df.columns:
                review_col = col
                break
        
        # Create standardized DataFrame
        if label_col and review_col:
            result_df = df[[review_col, label_col]].copy()
            result_df.columns = ['review', 'label']
        else:
            # Fallback: assume first column is review, last column is label
            result_df = df.iloc[:, [0, -1]].copy()
            result_df.columns = ['review', 'label']
    
    # Standardize label values (handle variations in capitalization and extra spaces)
    label_mapping = {
        'positive': 'Positive', 'POSITIVE': 'Positive', 'Positive': 'Positive',
        'negative': 'Negative', 'NEGATIVE': 'Negative', 'Negative': 'Negative',
        'neutral': 'Neutral', 'NEUTRAL': 'Neutral', 'Neutral': 'Neutral'
    }
    
    # Clean label values (strip whitespace and convert to lowercase)
    # Keep original values BEFORE mapping for error reporting
    result_df['label_original'] = result_df['label'].copy()
    result_df['label'] = result_df['label'].str.strip().str.lower()
    
    # Report unmapped labels BEFORE mapping (so we can see the actual values)
    unmapped_mask = ~result_df['label'].isin(label_mapping.keys())
    unmapped_values = result_df[unmapped_mask]
    if len(unmapped_values) > 0:
        print(f"⚠ Warning: {len(unmapped_values)} rows have unmapped labels and will be dropped.")
        print(f"  Unique unmapped values: {unmapped_values['label_original'].unique().tolist()}")
    
    # Apply label mapping
    result_df['label'] = result_df['label'].map(label_mapping)
    
    # Drop any rows with missing values
    result_df = result_df.dropna()
    
    # Drop the temporary column
    if 'label_original' in result_df.columns:
        result_df = result_df.drop(columns=['label_original'])
    
    # Reset index
    result_df = result_df.reset_index(drop=True)
    
    return result_df


#### 🔧 HELPER: Create Sample Annotator Files

**Run the cell below** to create sample annotator CSV files from your dataset.

⚠️ Replace these with real Label Studio exports for final submission!

In [14]:
# Helper: Create sample annotator CSV files from your movie reviews dataset
import pandas as pd
import numpy as np
import os

# Check if the movie reviews file exists
csv_filename = 'Movie_review - unique_movie_reviews.csv'

if os.path.exists(csv_filename):
    # Load the movie reviews
    df_movies = pd.read_csv(csv_filename)
    print(f"✓ Loaded {len(df_movies)} reviews from '{csv_filename}'")
    
    # Save as movie_reviews_300.csv for consistency
    df_movies.to_csv('movie_reviews_300.csv', index=False)
    print("✓ Saved as 'movie_reviews_300.csv'")
    
    # Use first 100 reviews for annotation simulation
    reviews_100 = df_movies['review'].iloc[:100].values
    
    # Generate base labels by analyzing keywords in reviews
    np.random.seed(42)
    base_labels = []
    
    for review in reviews_100:
        review_lower = str(review).lower()
        
        # Simple sentiment detection
        positive_words = ['amazing', 'love', 'best', 'excellent', 'fantastic', 'great', 
                         'wonderful', 'perfect', 'superb', 'masterpiece', 'triumph', 'joy']
        negative_words = ['terrible', 'worst', 'awful', 'horrible', 'disappointing', 
                         'boring', 'bad', 'waste', 'garbage', 'train wreck', 'bored']
        
        pos_score = sum(1 for word in positive_words if word in review_lower)
        neg_score = sum(1 for word in negative_words if word in review_lower)
        
        if pos_score > neg_score + 1:
            base_labels.append('Positive')
        elif neg_score > pos_score + 1:
            base_labels.append('Negative')
        else:
            base_labels.append('Neutral')
    
    # Annotator A: 10% disagreement
    labels_a = base_labels.copy()
    for idx in np.random.choice(100, size=10, replace=False):
        labels_a[idx] = 'Neutral' if labels_a[idx] != 'Neutral' else 'Positive'
    
    pd.DataFrame({'review': reviews_100, 'label': labels_a}).to_csv('annotator_a.csv', index=False)
    print(f"✓ Created 'annotator_a.csv': {pd.Series(labels_a).value_counts().to_dict()}")
    
    # Annotator B: 12% disagreement
    labels_b = base_labels.copy()
    for idx in np.random.choice(100, size=12, replace=False):
        if labels_b[idx] == 'Positive':
            labels_b[idx] = np.random.choice(['Neutral', 'Negative'])
        elif labels_b[idx] == 'Negative':
            labels_b[idx] = np.random.choice(['Neutral', 'Positive'])
        else:
            labels_b[idx] = np.random.choice(['Positive', 'Negative'])
    
    pd.DataFrame({'review': reviews_100, 'label': labels_b}).to_csv('annotator_b.csv', index=False)
    print(f"✓ Created 'annotator_b.csv': {pd.Series(labels_b).value_counts().to_dict()}")
    
    # Annotator C: 15% disagreement, prefers Neutral
    labels_c = base_labels.copy()
    for idx in np.random.choice(100, size=15, replace=False):
        labels_c[idx] = 'Neutral' if np.random.rand() > 0.3 else ('Negative' if labels_c[idx] == 'Positive' else 'Positive')
    
    pd.DataFrame({'review': reviews_100, 'label': labels_c}).to_csv('annotator_c.csv', index=False)
    print(f"✓ Created 'annotator_c.csv': {pd.Series(labels_c).value_counts().to_dict()}")
    
    print("\n" + "="*80)
    print("✅ All sample CSV files created! Run the next cell to parse them.")
    print("="*80)
    
else:
    print(f"⚠ Error: '{csv_filename}' not found.")
    print("Please ensure the file is in the same folder as this notebook.")


✓ Loaded 320 reviews from 'Movie_review - unique_movie_reviews.csv'
✓ Saved as 'movie_reviews_300.csv'
✓ Created 'annotator_a.csv': {'Neutral': 83, 'Positive': 15, 'Negative': 2}
✓ Created 'annotator_b.csv': {'Neutral': 84, np.str_('Positive'): 8, np.str_('Negative'): 8}
✓ Created 'annotator_c.csv': {'Neutral': 89, 'Positive': 9, 'Negative': 2}

✅ All sample CSV files created! Run the next cell to parse them.


In [15]:
# Parse CSV files from three annotators
# Note: Replace these paths with your actual annotator CSV file paths
# Example: df_a = parse_annotator_csv('annotator_a.csv')

try:
    df_a = parse_annotator_csv('annotator_a.csv')
    df_b = parse_annotator_csv('annotator_b.csv')
    df_c = parse_annotator_csv('annotator_c.csv')
    
    # Display sample data from each annotator
    print("=" * 80)
    print("ANNOTATOR A - First 3 rows:")
    print("=" * 80)
    print(df_a.head(3))
    print(f"\nTotal reviews annotated by A: {len(df_a)}")
    
    print("\n" + "=" * 80)
    print("ANNOTATOR B - First 3 rows:")
    print("=" * 80)
    print(df_b.head(3))
    print(f"\nTotal reviews annotated by B: {len(df_b)}")
    
    print("\n" + "=" * 80)
    print("ANNOTATOR C - First 3 rows:")
    print("=" * 80)
    print(df_c.head(3))
    print(f"\nTotal reviews annotated by C: {len(df_c)}")
    
    # Display label distribution for each annotator
    print("\n" + "=" * 80)
    print("LABEL DISTRIBUTION:")
    print("=" * 80)
    print("\nAnnotator A:")
    print(df_a['label'].value_counts())
    print("\nAnnotator B:")
    print(df_b['label'].value_counts())
    print("\nAnnotator C:")
    print(df_c['label'].value_counts())
    
except FileNotFoundError as e:
    print(f"⚠ Error: Annotator CSV file not found: {e}")
    print("Please ensure annotator_a.csv, annotator_b.csv, and annotator_c.csv are in the current directory.")
    print("You should export these files from Label Studio after completing annotations.")


ANNOTATOR A - First 3 rows:
                                              review     label
0  This movie is a triumph in every sense. Highly...  Positive
1  I have never been so bored in my life. The sco...   Neutral
2  I was completely blown away by this film. The ...   Neutral

Total reviews annotated by A: 100

ANNOTATOR B - First 3 rows:
                                              review    label
0  This movie is a triumph in every sense. Highly...  Neutral
1  I have never been so bored in my life. The sco...  Neutral
2  I was completely blown away by this film. The ...  Neutral

Total reviews annotated by B: 100

ANNOTATOR C - First 3 rows:
                                              review    label
0  This movie is a triumph in every sense. Highly...  Neutral
1  I have never been so bored in my life. The sco...  Neutral
2  I was completely blown away by this film. The ...  Neutral

Total reviews annotated by C: 100

LABEL DISTRIBUTION:

Annotator A:
label
Neutral     83
Posit

### Part 1.2: Implement Fleiss' Kappa from Scratch

Measure inter-annotator agreement using Fleiss' Kappa statistic.
Implement the formula from scratch and compare with statsmodels implementation.

In [16]:
def fleiss_kappa_scratch(rating_matrix):
    """
    Computes Fleiss' Kappa for multiple raters from scratch.
    
    Args:
        rating_matrix (np.array): A Count Matrix of shape (N, k).
                                  - N = number of items (rows)
                                  - k = number of categories (columns)
                                  - Element [i, j] = Count of raters who assigned category j to item i.
                                  Example: 
                                    [[0, 0, 3],   # Item 0: All 3 raters said Category 2
                                     [1, 2, 0]]   # Item 1: 1 rater said Cat 0, 2 said Cat 1
                            
    
    Returns:
        float: Kappa score (ranges from -1 to 1, where 1 = perfect agreement)
    
    Formula:
        κ = (P_bar - P_e_bar) / (1 - P_e_bar)
        
        where:
        - P_bar = (1/N) * Σ(P_i) = average proportion of agreement across all items
        - P_i = (1/(n*(n-1))) * Σ(k_ij * (k_ij - 1)) for item i
        - P_e_bar = Σ(p_j^2) = expected agreement by chance
        - p_j = proportion of all assignments to category j
    
    Note:
        - N = number of items (samples)
        - n = number of raters per item (should be constant)
        - k_ij = number of raters who assigned category j to item i
    """
    # Get dimensions
    N = rating_matrix.shape[0]  # Number of items (samples)
    k = rating_matrix.shape[1]  # Number of categories
    n = rating_matrix.sum(axis=1)[0]  # Number of raters per item (should be constant)
    
    # Step 1: Calculate P_i for each item
    # P_i = (1/(n*(n-1))) * sum(k_ij * (k_ij - 1)) where k_ij is count for category j on item i
    P_i_values = []
    for i in range(N):
        # Get counts for this item across all categories
        counts = rating_matrix[i, :]
        # Calculate sum of k_ij * (k_ij - 1)
        sum_term = np.sum(counts * (counts - 1))
        # Calculate P_i for this item
        P_i = sum_term / (n * (n - 1))
        P_i_values.append(P_i)
    
    # Step 2: Calculate P_bar (observed agreement)
    # P_bar = average of all P_i values
    P_bar = np.mean(P_i_values)
    
    # Step 3: Calculate P_e_bar (expected agreement by chance)
    # p_j = proportion of all assignments to category j
    # P_e_bar = sum(p_j^2) for all categories
    
    # Calculate total number of assignments across all items
    total_assignments = N * n
    
    # Calculate p_j for each category (proportion of assignments to category j)
    p_j_values = []
    for j in range(k):
        # Sum of all assignments to category j across all items
        category_j_count = rating_matrix[:, j].sum()
        # Proportion
        p_j = category_j_count / total_assignments
        p_j_values.append(p_j)
    
    # Calculate P_e_bar = sum of squared proportions
    P_e_bar = np.sum(np.array(p_j_values) ** 2)
    
    # Step 4: Apply Fleiss' Kappa formula
    # κ = (P_bar - P_e_bar) / (1 - P_e_bar)
    if P_e_bar == 1:
        # Edge case: if P_e_bar = 1, denominator is 0
        return 0.0
    
    kappa = (P_bar - P_e_bar) / (1 - P_e_bar)
    
    return kappa


In [6]:
def prepare_rating_matrix(df_a, df_b, df_c):
    """
    Converts three DataFrames into a rating matrix for Fleiss' Kappa calculation.
    
    Args:
        df_a, df_b, df_c: DataFrames with columns ['review', 'label']
        
    Returns:
        np.array: Rating matrix of shape (N_samples, N_categories)
                  where categories are ['Negative', 'Neutral', 'Positive']
    """
    # Merge the three DataFrames on review column
    # Use suffixes to distinguish labels from different annotators
    merged = df_a.merge(df_b, on='review', suffixes=('_a', '_b'))
    merged = merged.merge(df_c, on='review')
    merged.columns = ['review', 'label_a', 'label_b', 'label_c']
    
    # Define category order: [Negative, Neutral, Positive]
    categories = ['Negative', 'Neutral', 'Positive']
    
    # Initialize rating matrix (N_samples x 3 categories)
    N = len(merged)
    rating_matrix = np.zeros((N, 3), dtype=int)
    
    # For each sample, count how many raters assigned each category
    for i in range(N):
        labels = [merged.iloc[i]['label_a'], 
                  merged.iloc[i]['label_b'], 
                  merged.iloc[i]['label_c']]
        
        # Count occurrences of each category
        for j, category in enumerate(categories):
            rating_matrix[i, j] = labels.count(category)
    
    return rating_matrix

# Prepare rating matrix for Fleiss' Kappa calculation
try:
    # Create rating matrix from the three annotators
    rating_matrix = prepare_rating_matrix(df_a, df_b, df_c)
    
    print("=" * 80)
    print("RATING MATRIX SAMPLE (First 5 rows):")
    print("=" * 80)
    print("Each row represents one review.")
    print("Columns: [Negative_count, Neutral_count, Positive_count]")
    print(f"Shape: {rating_matrix.shape} (N_samples={rating_matrix.shape[0]}, N_categories={rating_matrix.shape[1]})")
    print("\n", rating_matrix[:5])
    
    # Calculate Fleiss' Kappa using our implementation
    kappa_scratch = fleiss_kappa_scratch(rating_matrix)
    print("\n" + "=" * 80)
    print("FLEISS' KAPPA (Our Implementation):")
    print("=" * 80)
    print(f"Kappa Score: {kappa_scratch:.4f}")
    
    # Interpret the kappa score
    print("\nInterpretation:")
    if kappa_scratch < 0:
        print("  → Poor agreement (less than chance)")
    elif kappa_scratch < 0.2:
        print("  → Slight agreement")
    elif kappa_scratch < 0.4:
        print("  → Fair agreement")
    elif kappa_scratch < 0.6:
        print("  → Moderate agreement")
    elif kappa_scratch < 0.8:
        print("  → Substantial agreement")
    else:
        print("  → Almost perfect agreement")
    
    # Use statsmodels to calculate Fleiss' Kappa for comparison
    kappa_statsmodels = fleiss_kappa(rating_matrix, method='fleiss')
    print("\n" + "=" * 80)
    print("FLEISS' KAPPA (Statsmodels Implementation):")
    print("=" * 80)
    print(f"Kappa Score: {kappa_statsmodels:.4f}")
    
    # Print the difference between implementations
    difference = abs(kappa_scratch - kappa_statsmodels)
    print("\n" + "=" * 80)
    print("COMPARISON:")
    print("=" * 80)
    print(f"Our Implementation:     {kappa_scratch:.6f}")
    print(f"Statsmodels:            {kappa_statsmodels:.6f}")
    print(f"Absolute Difference:    {difference:.6f}")
    
    if difference < 0.001:
        print("✓ Implementation verified! (Difference < 0.001)")
    else:
        print("⚠ Implementations differ slightly. Check calculation logic.")
        
except NameError:
    print("⚠ Error: Please run the previous cell to parse annotator CSV files first.")


⚠ Error: Please run the previous cell to parse annotator CSV files first.


### Part 1.3: Conflict Resolution

Identify conflicts where annotators disagree and resolve them using majority vote.
For complete ties (all three differ), default to 'Neutral'.

In [17]:
def resolve_conflicts(df_a, df_b, df_c):
    """
    Merges annotations from 3 annotators, resolves disagreements using Majority Vote,
    and handles complete ties by defaulting to 'Neutral'.
    
    Args:
        df_a, df_b, df_c: DataFrames from each annotator with columns ['review', 'label']
        
    Returns:
        pd.DataFrame: Final DataFrame with resolved labels (gold standard)
                     Columns: ['review', 'label']
    
    Logic:
        - Majority Vote: If 2 annotators agree, use their label
        - Tie-Breaker: If all 3 differ (e.g., Positive vs. Negative vs. Neutral), assign 'Neutral'
    """
    # Merge all three annotators on review column
    merged = df_a.merge(df_b, on='review', suffixes=('_a', '_b'))
    merged = merged.merge(df_c, on='review')
    merged.columns = ['review', 'label_a', 'label_b', 'label_c']
    
    # Initialize list to store resolved labels
    resolved_labels = []
    
    # Iterate through each review and apply resolution logic
    for idx, row in merged.iterrows():
        labels = [row['label_a'], row['label_b'], row['label_c']]
        
        # Count occurrences of each label
        label_counts = {}
        for label in labels:
            label_counts[label] = label_counts.get(label, 0) + 1
        
        # Find the maximum count
        max_count = max(label_counts.values())
        
        # If majority exists (at least 2 agree), use that label
        if max_count >= 2:
            # Get the label with maximum count
            for label, count in label_counts.items():
                if count == max_count:
                    resolved_labels.append(label)
                    break
        else:
            # Complete tie: all three annotators chose different labels
            # Default to 'Neutral' as per assignment requirements
            resolved_labels.append('Neutral')
    
    # Create result DataFrame
    result = pd.DataFrame({
        'review': merged['review'],
        'label': resolved_labels
    })
    
    return result, merged  # Return both resolved and merged for conflict analysis


In [18]:
# Resolve conflicts and create gold standard
try:
    gold_standard, merged_annotations = resolve_conflicts(df_a, df_b, df_c)
    
    print("=" * 80)
    print("GOLD STANDARD CREATION SUMMARY:")
    print("=" * 80)
    print(f"Total reviews: {len(gold_standard)}")
    print(f"\nLabel distribution in Gold Standard:")
    print(gold_standard['label'].value_counts())
    
    # Identify conflicts: reviews where annotators did not unanimously agree
    conflicts = []
    for idx, row in merged_annotations.iterrows():
        labels = [row['label_a'], row['label_b'], row['label_c']]
        # Check if all three labels are not the same
        if len(set(labels)) > 1:
            conflicts.append({
                'review': row['review'],
                'annotator_a': row['label_a'],
                'annotator_b': row['label_b'],
                'annotator_c': row['label_c'],
                'resolved': gold_standard.iloc[idx]['label']
            })
    
    print(f"\n" + "=" * 80)
    print(f"CONFLICT ANALYSIS:")
    print("=" * 80)
    print(f"Total conflicts (disagreements): {len(conflicts)}")
    print(f"Conflict rate: {len(conflicts)/len(gold_standard)*100:.2f}%")
    
    # Display 5 examples of conflicting reviews (or all if fewer than 5)
    print(f"\n" + "=" * 80)
    print(f"SAMPLE CONFLICTING REVIEWS (showing up to 5):")
    print("=" * 80)
    
    num_to_show = min(5, len(conflicts))
    for i in range(num_to_show):
        conflict = conflicts[i]
        print(f"\n{i+1}. CONFLICT EXAMPLE:")
        print(f"   Review: {conflict['review'][:100]}...")  # Show first 100 chars
        print(f"   Annotator A: {conflict['annotator_a']}")
        print(f"   Annotator B: {conflict['annotator_b']}")
        print(f"   Annotator C: {conflict['annotator_c']}")
        print(f"   → Resolved: {conflict['resolved']}")
    
    # Save gold standard to CSV
    gold_standard.to_csv('gold_standard_100.csv', index=False)
    print("\n" + "=" * 80)
    print("✓ Gold standard saved to 'gold_standard_100.csv'")
    print("=" * 80)
    
    # Display first few rows of gold standard
    print("\nFirst 5 rows of Gold Standard:")
    print(gold_standard.head())
    
except NameError:
    print("⚠ Error: Please run the previous cells to parse annotator CSV files first.")


GOLD STANDARD CREATION SUMMARY:
Total reviews: 100

Label distribution in Gold Standard:
label
Neutral     93
Positive     5
Negative     2
Name: count, dtype: int64

CONFLICT ANALYSIS:
Total conflicts (disagreements): 26
Conflict rate: 26.00%

SAMPLE CONFLICTING REVIEWS (showing up to 5):

1. CONFLICT EXAMPLE:
   Review: This movie is a triumph in every sense. Highly recommended for everyone....
   Annotator A: Positive
   Annotator B: Neutral
   Annotator C: Neutral
   → Resolved: Neutral

2. CONFLICT EXAMPLE:
   Review: It perfectly balances humor and drama. I was hooked from the very first minute....
   Annotator A: Neutral
   Annotator B: Neutral
   Annotator C: Positive
   → Resolved: Neutral

3. CONFLICT EXAMPLE:
   Review: I oscillated between loving and hating this film. I respect the ambition, even if it didn't fully la...
   Annotator A: Positive
   Annotator B: Neutral
   Annotator C: Neutral
   → Resolved: Neutral

4. CONFLICT EXAMPLE:
   Review: The direction was typical,

## Task 2: Weak Supervision (The "Lazy" Labeler) (6 Marks)

**Objective:** Label the next 200 reviews programmatically to save time.

### Part 2.1: Heuristic Development

Analyze patterns in the gold standard and write at least 3 heuristic functions.
Apply them to the remaining 200 unlabeled reviews.

In [19]:
# Constants for labeling functions
POSITIVE = 1
NEGATIVE = 0
NEUTRAL = 2
ABSTAIN = -1

# Load gold standard to analyze patterns
try:
    df_gold = pd.read_csv('gold_standard_100.csv')
    print("=" * 80)
    print("ANALYZING GOLD STANDARD FOR PATTERNS:")
    print("=" * 80)
    print(f"Total samples: {len(df_gold)}")
    print(f"\nLabel distribution:")
    print(df_gold['label'].value_counts())
    
    # Analyze common words in each category
    print("\n" + "=" * 80)
    print("PATTERN ANALYSIS FOR HEURISTIC DESIGN:")
    print("=" * 80)
    
    # Sample positive and negative reviews
    positive_reviews = df_gold[df_gold['label'] == 'Positive']['review'].values
    negative_reviews = df_gold[df_gold['label'] == 'Negative']['review'].values
    neutral_reviews = df_gold[df_gold['label'] == 'Neutral']['review'].values if 'Neutral' in df_gold['label'].values else []
    
    print(f"\nPositive reviews: {len(positive_reviews)}")
    print(f"Negative reviews: {len(negative_reviews)}")
    print(f"Neutral reviews: {len(neutral_reviews)}")
    
    # Analyze review lengths
    print("\n--- Review Length Statistics ---")
    df_gold['review_length'] = df_gold['review'].str.len()
    print(df_gold.groupby('label')['review_length'].describe())
    
    print("\n--- Common Patterns to Consider ---")
    print("✓ Positive keywords: great, excellent, amazing, wonderful, perfect, love")
    print("✓ Negative keywords: bad, terrible, awful, horrible, worst, hate")
    print("✓ Neutral indicators: okay, fine, average, meh, decent")
    print("✓ Length: Very short reviews might indicate neutral sentiment")
    print("✓ Exclamation marks: Often indicate strong sentiment (positive or negative)")
    
except FileNotFoundError:
    print("⚠ Error: gold_standard_100.csv not found.")
    print("Please run Task 1 first to create the gold standard.")


ANALYZING GOLD STANDARD FOR PATTERNS:
Total samples: 100

Label distribution:
label
Neutral     93
Positive     5
Negative     2
Name: count, dtype: int64

PATTERN ANALYSIS FOR HEURISTIC DESIGN:

Positive reviews: 5
Negative reviews: 2
Neutral reviews: 93

--- Review Length Statistics ---
          count        mean        std   min    25%    50%     75%    max
label                                                                    
Negative    2.0   90.500000  41.719300  61.0  75.75   90.5  105.25  120.0
Neutral    93.0   94.311828  29.345757  34.0  72.00   95.0  110.00  181.0
Positive    5.0  107.600000  20.391175  78.0  95.00  117.0  123.00  125.0

--- Common Patterns to Consider ---
✓ Positive keywords: great, excellent, amazing, wonderful, perfect, love
✓ Negative keywords: bad, terrible, awful, horrible, worst, hate
✓ Neutral indicators: okay, fine, average, meh, decent
✓ Length: Very short reviews might indicate neutral sentiment
✓ Exclamation marks: Often indicate strong senti

### Part 2.2: Snorkel Labeling Functions

Wrap your heuristics as Snorkel @labeling_function decorators.
Each function should return POSITIVE (1), NEGATIVE (0), NEUTRAL (2), or ABSTAIN (-1).

In [20]:
@labeling_function()
def lf_keyword_great(x):
    """
    Labeling function: Check if "great" appears in the review.
    Returns POSITIVE if found, otherwise ABSTAIN.
    """
    # Check if "great" (case-insensitive) is in the review
    if 'great' in x.review.lower():
        return POSITIVE
    return ABSTAIN

@labeling_function()
def lf_short_review(x):
    """
    Label based on review length.
    Very short reviews might be neutral or indicate lack of engagement.
    """
    # If review is very short (less than 50 characters), likely neutral
    if len(x.review) < 50:
        return NEUTRAL
    return ABSTAIN

@labeling_function()
def lf_regex_bad(x):
    """
    Use regex to find negative patterns.
    Look for words like "horrible", "terrible", "awful", etc.
    """
    # Use regex or string matching to find negative keywords
    negative_pattern = r'\b(horrible|terrible|awful|worst|hate|hated|disappointing|disappointed)\b'
    if re.search(negative_pattern, x.review.lower()):
        return NEGATIVE
    return ABSTAIN

@labeling_function()
def lf_positive_words(x):
    """
    Detect multiple positive keywords in the review.
    """
    positive_keywords = ['excellent', 'amazing', 'wonderful', 'perfect', 'love', 'loved', 
                         'fantastic', 'brilliant', 'outstanding', 'superb']
    review_lower = x.review.lower()
    for keyword in positive_keywords:
        if keyword in review_lower:
            return POSITIVE
    return ABSTAIN

@labeling_function()
def lf_negative_words(x):
    """
    Detect negative keywords like "bad", "poor", "waste".
    """
    negative_keywords = ['bad', 'poor', 'waste', 'boring', 'dull', 'mediocre']
    review_lower = x.review.lower()
    for keyword in negative_keywords:
        if keyword in review_lower:
            return NEGATIVE
    return ABSTAIN

@labeling_function()
def lf_exclamation_marks(x):
    """
    Reviews with multiple exclamation marks often indicate strong positive sentiment.
    """
    exclamation_count = x.review.count('!')
    if exclamation_count >= 2:
        # Check if there are positive words nearby
        if any(word in x.review.lower() for word in ['great', 'love', 'amazing', 'excellent']):
            return POSITIVE
    return ABSTAIN

@labeling_function()
def lf_neutral_indicators(x):
    """
    Detect neutral language like "okay", "fine", "average".
    """
    neutral_keywords = ['okay', 'ok', 'fine', 'average', 'decent', 'meh', 'so-so']
    review_lower = x.review.lower()
    for keyword in neutral_keywords:
        if keyword in review_lower:
            return NEUTRAL
    return ABSTAIN

@labeling_function()
def lf_strong_negative(x):
    """
    Detect very strong negative sentiment with words like "never" or "don't".
    """
    strong_negative_pattern = r'\b(never|don\'t|do not|wouldn\'t|couldn\'t) (recommend|watch)\b'
    if re.search(strong_negative_pattern, x.review.lower()):
        return NEGATIVE
    return ABSTAIN

@labeling_function()
def lf_rating_mention(x):
    """
    Detect explicit rating mentions (e.g., "10/10", "5 stars").
    """
    # High ratings indicate positive sentiment
    if re.search(r'\b(10/10|5 stars?|five stars?)\b', x.review.lower()):
        return POSITIVE
    # Low ratings indicate negative sentiment
    if re.search(r'\b(1/10|0/10|1 star|zero stars?)\b', x.review.lower()):
        return NEGATIVE
    return ABSTAIN

print("=" * 80)
print("LABELING FUNCTIONS DEFINED:")
print("=" * 80)
print("✓ lf_keyword_great - Detects 'great' in reviews")
print("✓ lf_short_review - Identifies very short reviews as neutral")
print("✓ lf_regex_bad - Finds strong negative words")
print("✓ lf_positive_words - Detects positive keywords")
print("✓ lf_negative_words - Detects negative keywords")
print("✓ lf_exclamation_marks - Strong positive sentiment indicators")
print("✓ lf_neutral_indicators - Neutral language detection")
print("✓ lf_strong_negative - Very strong negative phrases")
print("✓ lf_rating_mention - Explicit rating mentions")
print("\nTotal: 9 labeling functions")


LABELING FUNCTIONS DEFINED:
✓ lf_keyword_great - Detects 'great' in reviews
✓ lf_short_review - Identifies very short reviews as neutral
✓ lf_regex_bad - Finds strong negative words
✓ lf_positive_words - Detects positive keywords
✓ lf_negative_words - Detects negative keywords
✓ lf_exclamation_marks - Strong positive sentiment indicators
✓ lf_neutral_indicators - Neutral language detection
✓ lf_strong_negative - Very strong negative phrases
✓ lf_rating_mention - Explicit rating mentions

Total: 9 labeling functions


def analyze_weak_labels(L_matrix, lfs):
    """
    Prints Coverage and Conflict statistics for the Labeling Functions.
    
    Args:
        L_matrix (np.array): Label matrix of shape (N_samples, N_functions)
                            Each column represents one labeling function's outputs
                            Values: POSITIVE (1), NEGATIVE (0), NEUTRAL (2), ABSTAIN (-1)
        lfs: List of labeling functions (for display names)
    
    Metrics to calculate:
        - Coverage: Percentage of non-abstain votes per LF
        - Conflict Rate: Percentage of samples where LFs disagree
    """
    print("=" * 80)
    print("LABELING FUNCTION ANALYSIS:")
    print("=" * 80)
    
    # Calculate coverage for each labeling function
    # Coverage = (number of non-abstain votes) / (total samples) * 100
    N_samples = L_matrix.shape[0]
    N_functions = L_matrix.shape[1]
    
    print("\n--- Coverage Statistics (per Labeling Function) ---")
    print(f"{'LF Name':<30} {'Coverage':<12} {'Positive':<10} {'Negative':<10} {'Neutral':<10}")
    print("-" * 80)
    
    for i, lf in enumerate(lfs):
        # Get the column for this labeling function
        lf_labels = L_matrix[:, i]
        
        # Count non-abstain votes
        non_abstain = np.sum(lf_labels != ABSTAIN)
        coverage = (non_abstain / N_samples) * 100
        
        # Count each label type
        positive_count = np.sum(lf_labels == POSITIVE)
        negative_count = np.sum(lf_labels == NEGATIVE)
        neutral_count = np.sum(lf_labels == NEUTRAL)
        
        print(f"{lf.__name__:<30} {coverage:>6.2f}%      {positive_count:<10} {negative_count:<10} {neutral_count:<10}")
    
    # Calculate overall coverage (percentage of samples with at least one non-abstain vote)
    has_label = np.any(L_matrix != ABSTAIN, axis=1)
    overall_coverage = (np.sum(has_label) / N_samples) * 100
    print("-" * 80)
    print(f"Overall Coverage: {overall_coverage:.2f}% ({np.sum(has_label)}/{N_samples} samples labeled by at least one LF)")
    
    # Calculate conflict rate
    # Conflict occurs when multiple LFs label the same sample differently (excluding abstentions)
    print("\n--- Conflict Analysis ---")
    conflicts = 0
    for i in range(N_samples):
        # Get all non-abstain labels for this sample
        sample_labels = L_matrix[i, :]
        non_abstain_labels = sample_labels[sample_labels != ABSTAIN]
        
        # If there are at least 2 non-abstain labels and they disagree, it's a conflict
        if len(non_abstain_labels) >= 2:
            unique_labels = np.unique(non_abstain_labels)
            if len(unique_labels) > 1:
                conflicts += 1
    
    conflict_rate = (conflicts / N_samples) * 100
    print(f"Conflict Rate: {conflict_rate:.2f}% ({conflicts}/{N_samples} samples have conflicting labels)")
    
    # Calculate agreement rate (samples where all non-abstain LFs agree)
    agreements = 0
    for i in range(N_samples):
        sample_labels = L_matrix[i, :]
        non_abstain_labels = sample_labels[sample_labels != ABSTAIN]
        if len(non_abstain_labels) >= 2:
            unique_labels = np.unique(non_abstain_labels)
            if len(unique_labels) == 1:
                agreements += 1
    
    agreement_rate = (agreements / N_samples) * 100
    print(f"Agreement Rate: {agreement_rate:.2f}% ({agreements}/{N_samples} samples where all LFs agree)")
    print("=" * 80)

# Load the 200 unlabeled reviews
# We'll load the full movie_reviews_300.csv and take the last 200 reviews (rows 100-299)
try:
    # Load full dataset
    df_full = pd.read_csv('movie_reviews_300.csv')
    
    # Take the last 200 reviews (assuming first 100 were used for gold standard)
    df_unlabeled = df_full.iloc[100:300].copy()
    df_unlabeled = df_unlabeled.reset_index(drop=True)
    
    print("=" * 80)
    print("LOADED UNLABELED DATA:")
    print("=" * 80)
    print(f"Total unlabeled reviews: {len(df_unlabeled)}")
    print(f"\nFirst 3 unlabeled reviews:")
    print(df_unlabeled.head(3))
    
    # Apply all labeling functions to create L_matrix
    lfs = [lf_keyword_great, lf_short_review, lf_regex_bad, lf_positive_words, 
           lf_negative_words, lf_exclamation_marks, lf_neutral_indicators, 
           lf_strong_negative, lf_rating_mention]
    
    print(f"\nApplying {len(lfs)} labeling functions...")
    applier = PandasLFApplier(lfs=lfs)
    L_matrix = applier.apply(df=df_unlabeled)
    
    print(f"✓ Label matrix created: shape {L_matrix.shape}")
    print(f"  (N_samples={L_matrix.shape[0]}, N_labeling_functions={L_matrix.shape[1]})")
    
    # Analyze coverage and conflicts
    analyze_weak_labels(L_matrix, lfs)
    
    # Use LFAnalysis for detailed statistics
    print("\n" + "=" * 80)
    print("DETAILED LF ANALYSIS (using Snorkel's LFAnalysis):")
    print("=" * 80)
    lf_analysis = LFAnalysis(L=L_matrix, lfs=lfs)
    print(lf_analysis.lf_summary())
    
except FileNotFoundError as e:
    print(f"⚠ Error: File not found: {e}")
    print("Please ensure movie_reviews_300.csv is in the current directory.")


In [21]:
def analyze_weak_labels(L_matrix, lfs):
    """
    Prints Coverage and Conflict statistics for the Labeling Functions.
    
    Args:
        L_matrix (np.array): Label matrix of shape (N_samples, N_functions)
                            Each column represents one labeling function's outputs
                            Values: POSITIVE (1), NEGATIVE (0), NEUTRAL (2), ABSTAIN (-1)
        lfs: List of labeling functions (for display names)
    
    Metrics to calculate:
        - Coverage: Percentage of non-abstain votes per LF
        - Conflict Rate: Percentage of samples where LFs disagree
    """
    # TODO: Calculate coverage for each labeling function
    # Coverage = (number of non-abstain votes) / (total samples) * 100

    
    # TODO: Calculate conflict rate
    # Conflict occurs when multiple LFs label the same sample differently
    # Conflict Rate = (number of conflicting samples) / (total samples) * 100
 
    
    # TODO: Print statistics in a readable format
    # Hint: Use LFAnalysis from snorkel for detailed stats (optional)
    # Or print manually: LF name, Coverage %, Conflicts count
    
    pass

# TODO: Load the 200 unlabeled reviews (you can load the entire dataset and then filter as per the requirement)


# TODO: Apply all labeling functions to create L_matrix
# lfs = [lf_keyword_great, lf_short_review, lf_regex_bad, ...]  # Add all your LFs
# applier = <put your code here>
# L_matrix = <put your code here>

# TODO: Analyze coverage and conflicts


# TODO: Use LFAnalysis for detailed statistics

### Part 2.4: Majority Vote Adjudication

Use majority vote to generate probabilistic labels (weak labels) for the 200 reviews.
Save the result to `weak_labels_200.csv`.

In [22]:
# Train LabelModel to get probabilistic labels
try:
    print("=" * 80)
    print("TRAINING LABEL MODEL (Snorkel):")
    print("=" * 80)
    
    # Initialize and train the label model
    # The label model learns to weight and combine labeling functions
    label_model = LabelModel(cardinality=3, verbose=True)  # 3 classes: Negative (0), Positive (1), Neutral (2)
    label_model.fit(L_train=L_matrix, n_epochs=500, lr=0.001, log_freq=100, seed=42)
    
    # Get probabilistic labels
    # Returns probabilities for each class: [P(Negative), P(Positive), P(Neutral)]
    probs = label_model.predict_proba(L=L_matrix)
    
    # Get hard labels (argmax of probabilities)
    weak_labels_numeric = label_model.predict(L=L_matrix)
    
    print("\n" + "=" * 80)
    print("LABEL MODEL PREDICTIONS:")
    print("=" * 80)
    print(f"Shape of probability matrix: {probs.shape}")
    print(f"First 5 probability vectors:")
    print(probs[:5])
    
    # Convert numeric labels to text labels
    # Label mapping: 0 -> 'Negative', 1 -> 'Positive', 2 -> 'Neutral'
    label_mapping_inverse = {0: 'Negative', 1: 'Positive', 2: 'Neutral'}
    weak_labels_text = [label_mapping_inverse[label] for label in weak_labels_numeric]
    
    print(f"\nFirst 10 predicted labels: {weak_labels_text[:10]}")
    
    # Count label distribution
    from collections import Counter
    label_counts = Counter(weak_labels_text)
    print(f"\nWeak label distribution:")
    for label, count in label_counts.items():
        print(f"  {label}: {count} ({count/len(weak_labels_text)*100:.1f}%)")
    
    # Create DataFrame with reviews and weak labels
    df_weak_labels = pd.DataFrame({
        'review': df_unlabeled['review'],
        'label': weak_labels_text
    })
    
    # Also add numeric labels and confidence scores for reference
    df_weak_labels['label_numeric'] = weak_labels_numeric
    df_weak_labels['confidence'] = np.max(probs, axis=1)
    
    print("\n" + "=" * 80)
    print("WEAK LABELS DATAFRAME (First 5 rows):")
    print("=" * 80)
    print(df_weak_labels.head())
    
    # Analyze confidence
    print(f"\nConfidence statistics:")
    print(f"  Mean confidence: {df_weak_labels['confidence'].mean():.3f}")
    print(f"  Median confidence: {df_weak_labels['confidence'].median():.3f}")
    print(f"  Min confidence: {df_weak_labels['confidence'].min():.3f}")
    print(f"  Max confidence: {df_weak_labels['confidence'].max():.3f}")
    
    # Count low confidence predictions (< 0.5)
    low_confidence = np.sum(df_weak_labels['confidence'] < 0.5)
    print(f"  Low confidence predictions (< 0.5): {low_confidence} ({low_confidence/len(df_weak_labels)*100:.1f}%)")
    
    # Save to CSV (without confidence scores for submission)
    df_weak_labels[['review', 'label']].to_csv('weak_labels_200.csv', index=False)
    
    print("\n" + "=" * 80)
    print("✓ Weak labels saved to 'weak_labels_200.csv'")
    print("=" * 80)
    print(f"Total reviews labeled: {len(df_weak_labels)}")
    
except NameError as e:
    print(f"⚠ Error: {e}")
    print("Please run the previous cells to create L_matrix first.")


TRAINING LABEL MODEL (Snorkel):
⚠ Error: name 'L_matrix' is not defined
Please run the previous cells to create L_matrix first.


## Task 3: Active Learning (The Budget Optimizer) (5 Marks)

**Objective:** Simulate cost savings by training a model iteratively.

### Part 3.1: Query Strategy Implementation

Implement Least Confidence and Entropy Sampling from scratch.
These strategies select the most informative samples for labeling.

In [23]:
def least_confidence_sampling(model, X_pool, n_instances=10):
    """
    Selects samples where the model is least confident (uncertainty sampling).
    
    Args:
        model: Trained classifier with predict_proba() method
        X_pool: Feature matrix of unlabeled samples
        n_instances: Number of samples to select
    
    Returns:
        np.array: Indices of selected samples
        
    Strategy:
        Uncertainty = 1 - max(probability) across all classes
        For 3-class classification: Get probabilities for [Negative, Positive, Neutral]
        Select samples with highest uncertainty (lowest max probability)
    """
    # Get probability predictions from model
    # Shape: (N_samples, N_classes) where N_classes = 3
    probabilities = model.predict_proba(X_pool)
    
    # Calculate uncertainty: 1 - max(probability) for each sample
    # The max probability represents model's confidence, so 1-max gives uncertainty
    max_probs = np.max(probabilities, axis=1)
    uncertainties = 1 - max_probs
    
    # Select top n_instances samples with highest uncertainty
    # Use argsort to get indices sorted by uncertainty (descending order)
    # argsort gives ascending order, so we use negative to get descending
    most_uncertain_indices = np.argsort(-uncertainties)[:n_instances]
    
    return most_uncertain_indices

def entropy_sampling(model, X_pool, n_instances=10):
    """
    Selects samples with highest entropy (information gain).
    
    Args:
        model: Trained classifier with predict_proba() method
        X_pool: Feature matrix of unlabeled samples
        n_instances: Number of samples to select
    
    Returns:
        np.array: Indices of selected samples
        
    Strategy:
        Entropy = -sum(p * log(p)) for all classes
        For 3-class classification: Calculate entropy across [Negative, Positive, Neutral] probabilities
        Select samples with highest entropy (most uncertain across all classes)
    """
    # Get probability predictions from model
    probabilities = model.predict_proba(X_pool)
    
    # Calculate entropy: -sum(p * log(p)) for each sample
    # Add small epsilon (1e-9) to avoid log(0) errors
    epsilon = 1e-9
    entropies = -np.sum(probabilities * np.log(probabilities + epsilon), axis=1)
    
    # Select top n_instances samples with highest entropy
    # Higher entropy = more uncertainty across all classes
    highest_entropy_indices = np.argsort(-entropies)[:n_instances]
    
    return highest_entropy_indices

def random_sampling(model, X_pool, n_instances=10):
    """
    Baseline strategy: Selects random samples.
    
    Args:
        model: Not used, but kept for interface consistency
        X_pool: Feature matrix of unlabeled samples
        n_instances: Number of samples to select
    
    Returns:
        np.array: Randomly selected indices
    """
    # Randomly select n_instances indices from X_pool
    n_samples = X_pool.shape[0]
    random_indices = np.random.choice(n_samples, size=n_instances, replace=False)
    
    return random_indices

print("=" * 80)
print("QUERY STRATEGIES IMPLEMENTED:")
print("=" * 80)
print("✓ least_confidence_sampling - Selects samples with lowest max probability")
print("✓ entropy_sampling - Selects samples with highest entropy")
print("✓ random_sampling - Baseline: random selection")
print("=" * 80)


QUERY STRATEGIES IMPLEMENTED:
✓ least_confidence_sampling - Selects samples with lowest max probability
✓ entropy_sampling - Selects samples with highest entropy
✓ random_sampling - Baseline: random selection


### Part 3.2: Data Processing and Setup

Load the gold standard (seed) and weak labels (pool).
Create a static test set from the pool for evaluation.
Vectorize text data using TF-IDF.

In [24]:
def load_and_process_data():
    """
    Loads and processes data for active learning.
    
    Returns:
        Tuple: (X_seed, y_seed, X_pool, y_pool, X_test, y_test, vectorizer)
               All X are feature matrices, all y are label arrays
               vectorizer is returned for later use on LLM data
               
    Note:
        - Seed: gold_standard_100.csv (100 labeled reviews)
        - Pool: weak_labels_200.csv (200 reviews, labels treated as hidden for simulation)
        - Test: Hold out 50 samples from pool (weak labels) for static evaluation
        - We use 3-class classification: Positive (1), Negative (0), Neutral (2)
        - Uncertainty metrics use probability scores across all three classes:
          * Least Confidence: 1 - max(probabilities) across all classes
          * Entropy: -sum(p * log(p)) for all three classes
    """

    df_seed = pd.read_csv('gold_standard_100.csv')
    df_pool_full = pd.read_csv('weak_labels_200.csv')
    
    # Ensure both have 'review' column
    if 'review' not in df_seed.columns:
        raise ValueError("gold_standard_100.csv must have 'review' column")
    if 'review' not in df_pool_full.columns:
        raise ValueError("weak_labels_200.csv must have 'review' column")
    
    # Handle both 'label' and 'sentiment' column names
    label_col_seed = 'label' if 'label' in df_seed.columns else 'sentiment'
    label_col_pool = 'label' if 'label' in df_pool_full.columns else 'sentiment'
    
    # Map text labels to numeric: Positive=1, Negative=0, Neutral=2
    label_mapping = {
        'Positive': 1, 'positive': 1, 'POSITIVE': 1,
        'Negative': 0, 'negative': 0, 'NEGATIVE': 0,
        'Neutral': 2, 'neutral': 2, 'NEUTRAL': 2
    }
    
    # Convert seed labels
    if df_seed[label_col_seed].dtype == 'object':
        df_seed['sentiment_numeric'] = df_seed[label_col_seed].map(label_mapping)
        if df_seed['sentiment_numeric'].isna().any():
            raise ValueError(f"Unknown labels in seed data: {df_seed[df_seed['sentiment_numeric'].isna()][label_col_seed].unique()}")
    else:
        df_seed['sentiment_numeric'] = df_seed[label_col_seed].values
    
    # Convert pool labels
    if df_pool_full[label_col_pool].dtype == 'object':
        df_pool_full['sentiment_numeric'] = df_pool_full[label_col_pool].map(label_mapping)
        if df_pool_full['sentiment_numeric'].isna().any():
            raise ValueError(f"Unknown labels in pool data: {df_pool_full[df_pool_full['sentiment_numeric'].isna()][label_col_pool].unique()}")
    else:
        df_pool_full['sentiment_numeric'] = df_pool_full[label_col_pool].values
    
    # Create static test set (hold out 50 samples from pool)
    df_pool, df_test = train_test_split(df_pool_full, test_size=50, random_state=42)
    
    # Vectorize text data using TfidfVectorizer
    # Fit vectorizer on ALL text (seed + pool + test) to ensure consistent dimensions
    vectorizer = TfidfVectorizer(stop_words='english', max_features=1000)
    all_text = pd.concat([df_seed['review'], df_pool['review'], df_test['review']])
    vectorizer.fit(all_text)
    
    # Transform datasets to feature matrices
    X_seed = vectorizer.transform(df_seed['review']).toarray()
    X_pool = vectorizer.transform(df_pool['review']).toarray()
    X_test = vectorizer.transform(df_test['review']).toarray()
    
    # Extract numeric labels
    y_seed = df_seed['sentiment_numeric'].values
    y_pool = df_pool['sentiment_numeric'].values
    y_test = df_test['sentiment_numeric'].values
    
    # Return all datasets and vectorizer
    return X_seed, y_seed, X_pool, y_pool, X_test, y_test, vectorizer

# Load and process data for active learning
try:
    X_seed, y_seed, X_pool, y_pool, X_test, y_test, vectorizer = load_and_process_data()

    print("=" * 80)
    print("DATA LOADED AND PROCESSED:")
    print("=" * 80)
    print(f"Seed Set Size: {len(y_seed)} (Initial training data)")
    print(f"Pool Set Size: {len(y_pool)} (Available for querying)")
    print(f"Test Set Size: {len(y_test)} (Held out for evaluation)")
    print(f"\nFeature dimensions: {X_seed.shape[1]} (TF-IDF features)")
    
    print(f"\nSeed label distribution:")
    unique, counts = np.unique(y_seed, return_counts=True)
    for label, count in zip(unique, counts):
        label_name = {0: 'Negative', 1: 'Positive', 2: 'Neutral'}[label]
        print(f"  {label_name} ({label}): {count}")
    
    print(f"\nPool label distribution (hidden for simulation):")
    unique, counts = np.unique(y_pool, return_counts=True)
    for label, count in zip(unique, counts):
        label_name = {0: 'Negative', 1: 'Positive', 2: 'Neutral'}[label]
        print(f"  {label_name} ({label}): {count}")
    
    print("=" * 80)
    
except FileNotFoundError as e:
    print(f"⚠ Error: {e}")
    print("Please run previous tasks to create gold_standard_100.csv and weak_labels_200.csv")


⚠ Error: [Errno 2] No such file or directory: 'weak_labels_200.csv'
Please run previous tasks to create gold_standard_100.csv and weak_labels_200.csv


### Part 3.3: Active Learning Loop

Implement the iterative active learning loop:
1. Train model on current training set
2. Query uncertain samples from pool
3. "Label" them (reveal ground truth)
4. Add to training set and retrain
5. Log test accuracy

In [25]:
def run_active_learning_loop(X_seed, y_seed, X_pool, y_pool, X_test, y_test, 
                             strategy_func, steps=5, batch_size=10):
    """
    Simulates the active learning loop (matches lab approach).
    
    Args:
        X_seed, y_seed: Initial training data (seed set)
        X_pool, y_pool: Unlabeled pool (y_pool is hidden, revealed during query)
        X_test, y_test: Static test set for evaluation
        strategy_func: Function that selects samples (e.g., least_confidence_sampling)
                      Signature: strategy_func(model, X_pool, n_instances) -> indices
        steps: Number of iterations
        batch_size: Number of samples to query per iteration
    
    Returns:
        Tuple: (n_labels_history, accuracy_history)
               Lists tracking number of labels and test accuracy over iterations
    """
    # Initialize training set with seed data (make copies to avoid modifying originals)
    X_train = X_seed.copy()
    y_train = y_seed.copy()
    
    # Create working copies of pool (we'll remove samples as we query them)
    X_pool_curr = X_pool.copy()
    y_pool_curr = y_pool.copy()
    
    # Initialize empty lists to track progress
    accuracy_history = []
    n_labels_history = []
    
    # Train initial model on seed data
    model = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    
    # Evaluate initial model and log results
    y_pred = model.predict(X_test)
    initial_accuracy = accuracy_score(y_test, y_pred)
    accuracy_history.append(initial_accuracy)
    n_labels_history.append(len(y_train))
    
    print(f"Initial: {len(y_train)} labels, Test Accuracy: {initial_accuracy:.4f}")
    
    # Iterative loop (repeat 'steps' times)
    for iteration in range(steps):
        # Check if pool is exhausted
        if len(X_pool_curr) < batch_size:
            print(f"  ⚠ Pool exhausted at iteration {iteration+1}")
            break
        
        # 1. Query: Use strategy_func to select most informative samples
        query_indices = strategy_func(model, X_pool_curr, batch_size)
        
        # 2. "Label": Reveal ground truth from hidden labels
        X_new = X_pool_curr[query_indices]
        y_new = y_pool_curr[query_indices]
        
        # 3. Add to training set: append new samples
        X_train = np.vstack([X_train, X_new])
        y_train = np.concatenate([y_train, y_new])
        
        # 4. Remove from pool: delete queried samples
        X_pool_curr = np.delete(X_pool_curr, query_indices, axis=0)
        y_pool_curr = np.delete(y_pool_curr, query_indices, axis=0)
        
        # 5. Retrain model: update with new training data
        model.fit(X_train, y_train)
        
        # 6. Evaluate on test set
        y_pred = model.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)
        
        # 7. Log: track accuracy and number of labels
        accuracy_history.append(accuracy)
        n_labels_history.append(len(y_train))
        
        print(f"Iteration {iteration+1}: {len(y_train)} labels (+{batch_size}), Test Accuracy: {accuracy:.4f}")
    
    # Return history lists
    return n_labels_history, accuracy_history

# Run active learning with least confidence strategy
try:
    print("=" * 80)
    print("RUNNING ACTIVE LEARNING (Least Confidence Strategy):")
    print("=" * 80)
    
    # Set random seed for reproducibility
    np.random.seed(42)
    
    # Run active learning loop
    n_labels_lc, accuracy_lc = run_active_learning_loop(
        X_seed=X_seed,
        y_seed=y_seed,
        X_pool=X_pool,
        y_pool=y_pool,
        X_test=X_test,
        y_test=y_test,
        strategy_func=least_confidence_sampling,
        steps=5,
        batch_size=10
    )
    
    print("\n" + "=" * 80)
    print("ACTIVE LEARNING RESULTS (Least Confidence):")
    print("=" * 80)
    print(f"Final Training Set Size: {n_labels_lc[-1]}")
    print(f"Final Test Accuracy: {accuracy_lc[-1]:.4f}")
    print(f"Accuracy Improvement: {accuracy_lc[-1] - accuracy_lc[0]:.4f}")
    print("=" * 80)
    
except NameError as e:
    print(f"⚠ Error: {e}")
    print("Please run the previous cell to load and process data.")


RUNNING ACTIVE LEARNING (Least Confidence Strategy):
⚠ Error: name 'X_seed' is not defined
Please run the previous cell to load and process data.


### Part 3.4: Visualization and Comparison

Plot learning curves comparing Active Learning vs. Random Sampling.

In [26]:
# Run active learning with random sampling (baseline)
try:
    print("=" * 80)
    print("RUNNING BASELINE (Random Sampling Strategy):")
    print("=" * 80)
    
    # Set random seed for reproducibility
    np.random.seed(42)
    
    # Run active learning loop with random sampling
    n_labels_random, accuracy_random = run_active_learning_loop(
        X_seed=X_seed,
        y_seed=y_seed,
        X_pool=X_pool,
        y_pool=y_pool,
        X_test=X_test,
        y_test=y_test,
        strategy_func=random_sampling,
        steps=5,
        batch_size=10
    )
    
    print("\n" + "=" * 80)
    print("RANDOM SAMPLING RESULTS:")
    print("=" * 80)
    print(f"Final Training Set Size: {n_labels_random[-1]}")
    print(f"Final Test Accuracy: {accuracy_random[-1]:.4f}")
    print(f"Accuracy Improvement: {accuracy_random[-1] - accuracy_random[0]:.4f}")
    print("=" * 80)
    
    # Plot learning curves comparing Active Learning vs Random Sampling
    plt.figure(figsize=(12, 6))
    
    # Plot 1: Accuracy vs Number of Labels
    plt.subplot(1, 2, 1)
    plt.plot(n_labels_lc, accuracy_lc, marker='o', linewidth=2, markersize=8, 
             label='Active Learning (Least Confidence)', color='blue')
    plt.plot(n_labels_random, accuracy_random, marker='s', linewidth=2, markersize=8,
             label='Random Sampling (Baseline)', color='red', linestyle='--')
    plt.xlabel('Number of Labeled Samples', fontsize=12)
    plt.ylabel('Test Accuracy', fontsize=12)
    plt.title('Learning Curve: Active Learning vs Random Sampling', fontsize=14, fontweight='bold')
    plt.legend(fontsize=10)
    plt.grid(True, alpha=0.3)
    
    # Add annotations for final accuracies
    plt.annotate(f'Final: {accuracy_lc[-1]:.3f}', 
                xy=(n_labels_lc[-1], accuracy_lc[-1]), 
                xytext=(10, -10), textcoords='offset points',
                bbox=dict(boxstyle='round,pad=0.5', fc='blue', alpha=0.2),
                fontsize=9)
    plt.annotate(f'Final: {accuracy_random[-1]:.3f}', 
                xy=(n_labels_random[-1], accuracy_random[-1]), 
                xytext=(10, 10), textcoords='offset points',
                bbox=dict(boxstyle='round,pad=0.5', fc='red', alpha=0.2),
                fontsize=9)
    
    # Plot 2: Accuracy Improvement (relative to initial)
    plt.subplot(1, 2, 2)
    improvement_lc = [acc - accuracy_lc[0] for acc in accuracy_lc]
    improvement_random = [acc - accuracy_random[0] for acc in accuracy_random]
    
    plt.plot(n_labels_lc, improvement_lc, marker='o', linewidth=2, markersize=8,
             label='Active Learning', color='blue')
    plt.plot(n_labels_random, improvement_random, marker='s', linewidth=2, markersize=8,
             label='Random Sampling', color='red', linestyle='--')
    plt.xlabel('Number of Labeled Samples', fontsize=12)
    plt.ylabel('Accuracy Improvement (from initial)', fontsize=12)
    plt.title('Accuracy Improvement Over Iterations', fontsize=14, fontweight='bold')
    plt.legend(fontsize=10)
    plt.grid(True, alpha=0.3)
    plt.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
    
    plt.tight_layout()
    plt.savefig('active_learning_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    
    print("\n✓ Learning curve plot saved as 'active_learning_comparison.png'")
    
    # Print comparison summary
    print("\n" + "=" * 80)
    print("COMPARISON SUMMARY:")
    print("=" * 80)
    print(f"{'Metric':<35} {'Active Learning':<20} {'Random Sampling':<20}")
    print("-" * 80)
    print(f"{'Initial Accuracy':<35} {accuracy_lc[0]:<20.4f} {accuracy_random[0]:<20.4f}")
    print(f"{'Final Accuracy':<35} {accuracy_lc[-1]:<20.4f} {accuracy_random[-1]:<20.4f}")
    print(f"{'Accuracy Improvement':<35} {accuracy_lc[-1] - accuracy_lc[0]:<20.4f} {accuracy_random[-1] - accuracy_random[0]:<20.4f}")
    print(f"{'Labels Used':<35} {n_labels_lc[-1]:<20} {n_labels_random[-1]:<20}")
    
    # Calculate advantage
    advantage = accuracy_lc[-1] - accuracy_random[-1]
    print("-" * 80)
    print(f"Active Learning Advantage: {advantage:+.4f}")
    
    if advantage > 0:
        print(f"✓ Active Learning outperforms Random Sampling by {advantage:.4f}")
    elif advantage < 0:
        print(f"⚠ Random Sampling outperforms Active Learning by {-advantage:.4f}")
    else:
        print(f"= Both strategies achieved the same accuracy")
    
    print("=" * 80)
    
except NameError as e:
    print(f"⚠ Error: {e}")
    print("Please run the previous cells to perform active learning first.")


RUNNING BASELINE (Random Sampling Strategy):
⚠ Error: name 'X_seed' is not defined
Please run the previous cells to perform active learning first.


## Task 4: AI vs. AI (LLM & Noise Detection) (3 Marks)

**Objective:** Use LLMs for bulk labeling and detect hallucinations.

**Note:**

- Make an account at [open-router](https://openrouter.ai/) and get the API key.
- Use `google/gemini-2.5-flash-lite` (free tier) model as your LLM. Read the documentation on how to use it [here](https://openrouter.ai/google/gemini-2.5-flash-lite/api)
- Set environment variable using .env file and paste your API key in it.

### Part 4.1: LLM Pipeline with Few-Shot Prompting

Design a few-shot prompt with 3 examples from gold standard.
Send remaining unlabeled samples (~150) to Gemini API for labeling.

In [27]:

import os
import time
import json
import requests
import pandas as pd
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()
API_KEY = os.getenv('OPENROUTER_API_KEY')
SITE_URL = "http://localhost:8000"  # for OpenRouter rankings
SITE_NAME = "Student Lab Assignment"

MODEL_NAME = "google/gemini-2.5-flash-lite"

if not API_KEY:
    print("⚠ Warning: OPENROUTER_API_KEY not found. Please check your .env file.")
    print("\nTo set up:")
    print("1. Create an account at https://openrouter.ai/")
    print("2. Get your API key from the dashboard")
    print("3. Create a .env file in the current directory")
    print("4. Add this line: OPENROUTER_API_KEY=your_api_key_here")


def generate_few_shot_prompt(review_text, examples):
    """
    Constructs a few-shot prompt with 3 gold examples + target review.
    
    Args:
        review_text (str): The review to be labeled
        examples (list): List of 3 example dictionaries with 'review' and 'label' keys
    
    Returns:
        str: Formatted prompt string
    """
    # Create the system message and instructions
    prompt = """You are a sentiment analysis expert. Your task is to classify movie reviews into one of three categories: Positive, Negative, or Neutral.

Here are some examples to guide you:

"""
    
    # Add the 3 few-shot examples
    for i, example in enumerate(examples, 1):
        prompt += f"Example {i}:\n"
        prompt += f"Review: \"{example['review']}\"\n"
        prompt += f"Sentiment: {example['label']}\n\n"
    
    # Add the target review to classify
    prompt += f"Now classify this review:\n"
    prompt += f"Review: \"{review_text}\"\n"
    prompt += f"Sentiment: "
    
    # Add instructions for formatting
    prompt += "\n\nRespond with ONLY one word: Positive, Negative, or Neutral. Do not include any explanation or additional text."
    
    return prompt


def query_openrouter(review_text, examples):
    """
    Sends request to OpenRouter API with retry logic and parsing.
    
    Args:
        review_text (str): Review to classify
        examples (list): Few-shot examples (list of dicts with 'review' and 'label')
    
    Returns:
        str: Label ('Positive', 'Negative', or 'Neutral')
             Returns None if API fails or response is invalid
    
    Note:
        - Uses OpenRouter API endpoint: https://openrouter.ai/api/v1/chat/completions
        - Implements retry logic for rate limit errors (429)
        - Parses response from OpenRouter's chat completions format
    """
    url = "https://openrouter.ai/api/v1/chat/completions"
    
    # Set up headers with API key
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json",
        "HTTP-Referer": SITE_URL,
        "X-Title": SITE_NAME
    }
    
    # Generate prompt using few-shot examples
    prompt = generate_few_shot_prompt(review_text, examples)
    
    # Create payload dictionary
    payload = {
        "model": MODEL_NAME,
        "messages": [
            {"role": "user", "content": prompt}
        ],
        "temperature": 0.3,  # Lower temperature for more consistent responses
        "max_tokens": 10     # We only need one word
    }
    
    # Implement retry logic for rate limiting
    max_retries = 3
    retry_delay = 2  # seconds
    
    for attempt in range(max_retries):
        try:
            response = requests.post(url, headers=headers, json=payload, timeout=30)
            
            # Check for rate limit error (429)
            if response.status_code == 429:
                if attempt < max_retries - 1:
                    print(f"  ⚠ Rate limit hit, waiting {retry_delay} seconds...")
                    time.sleep(retry_delay)
                    retry_delay *= 2  # Exponential backoff
                    continue
                else:
                    print(f"  ✗ Rate limit exceeded after {max_retries} attempts")
                    return None
            
            # Check for successful response
            if response.status_code == 200:
                # Parse successful response
                result = response.json()
                
                # Extract the label from OpenRouter's response format
                if 'choices' in result and len(result['choices']) > 0:
                    content = result['choices'][0]['message']['content'].strip()
                    
                    # Clean up the response (remove extra text, normalize)
                    content_lower = content.lower()
                    if 'positive' in content_lower:
                        return 'Positive'
                    elif 'negative' in content_lower:
                        return 'Negative'
                    elif 'neutral' in content_lower:
                        return 'Neutral'
                    else:
                        print(f"  ⚠ Unexpected response format: {content}")
                        return 'Neutral'  # Default to Neutral for unclear responses
                else:
                    print(f"  ✗ Unexpected response structure")
                    return None
            else:
                print(f"  ✗ API error: {response.status_code} - {response.text}")
                return None
                
        except requests.exceptions.Timeout:
            print(f"  ⚠ Request timeout (attempt {attempt+1}/{max_retries})")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)
            else:
                return None
        except Exception as e:
            print(f"  ✗ Error: {e}")
            return None
    
    return None

# --- MAIN EXECUTION ---

# Load gold standard examples for few-shot prompting
try:
    df_gold = pd.read_csv('gold_standard_100.csv')
    
    # Select 3 diverse examples (1 Positive, 1 Negative, 1 Neutral if available)
    examples = []
    
    # Try to get one of each label
    for label in ['Positive', 'Negative', 'Neutral']:
        label_samples = df_gold[df_gold['label'] == label]
        if len(label_samples) > 0:
            # Pick a sample that's not too short and not too long (medium length)
            label_samples['review_len'] = label_samples['review'].str.len()
            median_idx = label_samples['review_len'].argsort().iloc[len(label_samples)//2]
            sample = label_samples.iloc[median_idx]
            examples.append({
                'review': sample['review'],
                'label': sample['label']
            })
    
    # If we don't have 3 examples yet (e.g., no Neutral), add more from available labels
    while len(examples) < 3:
        remaining = df_gold[~df_gold['review'].isin([e['review'] for e in examples])]
        if len(remaining) > 0:
            sample = remaining.iloc[0]
            examples.append({
                'review': sample['review'],
                'label': sample['label']
            })
        else:
            break
    
    print("=" * 80)
    print("FEW-SHOT EXAMPLES SELECTED:")
    print("=" * 80)
    for i, ex in enumerate(examples, 1):
        print(f"\nExample {i} ({ex['label']}):")
        print(f"  {ex['review'][:100]}...")
    print("=" * 80)
    
    # Load remaining unlabeled reviews (last 150 from movie_reviews_300.csv)
    df_full = pd.read_csv('movie_reviews_300.csv')
    df_to_label = df_full.iloc[150:300].copy()  # Last 150 reviews
    df_to_label = df_to_label.reset_index(drop=True)
    
    print(f"\nTotal reviews to label with LLM: {len(df_to_label)}")
    
    # Check if API key is available
    if not API_KEY:
        print("\n⚠ Cannot proceed without API key. Please set up your .env file.")
        print("Creating a dummy llm_labels_150.csv for demonstration...")
        # Create dummy data
        df_llm_labels = pd.DataFrame({
            'review': df_to_label['review'],
            'label': ['Neutral'] * len(df_to_label)  # Dummy labels
        })
        df_llm_labels.to_csv('llm_labels_150.csv', index=False)
        print("✓ Dummy file created. Please set up API key to get real LLM labels.")
    else:
        # Query OpenRouter for each review
        print("\n" + "=" * 80)
        print("QUERYING LLM (This may take a while due to rate limits):")
        print("=" * 80)
        
        llm_labels = []
        
        # Free tier usually has ~15 requests per minute limit
        # We'll add a small delay between requests to avoid hitting the limit
        request_delay = 4  # seconds between requests (15 requests/minute ≈ 4 seconds/request)
        
        for idx, row in df_to_label.iterrows():
            review_text = row['review']
            
            # Show progress
            print(f"\nLabeling {idx+1}/{len(df_to_label)}: {review_text[:60]}...")
            
            # Query the API
            label = query_openrouter(review_text, examples)
            
            if label:
                llm_labels.append(label)
                print(f"  ✓ Label: {label}")
            else:
                # If API fails, default to Neutral
                llm_labels.append('Neutral')
                print(f"  ⚠ API failed, defaulting to: Neutral")
            
            # Add delay to respect rate limits (except for last request)
            if idx < len(df_to_label) - 1:
                time.sleep(request_delay)
        
        # Save LLM labels in CSV format with 'review' and 'label' columns
        df_llm_labels = pd.DataFrame({
            'review': df_to_label['review'],
            'label': llm_labels
        })
        
        df_llm_labels.to_csv('llm_labels_150.csv', index=False)
        
        print("\n" + "=" * 80)
        print("✓ LLM labels saved to 'llm_labels_150.csv'")
        print("=" * 80)
        
        # Display statistics
        print(f"\nLabel distribution:")
        print(df_llm_labels['label'].value_counts())
        
        print(f"\nFirst 5 LLM-labeled reviews:")
        print(df_llm_labels.head())

except FileNotFoundError as e:
    print(f"⚠ Error: Required file not found: {e}")
    print("Please ensure gold_standard_100.csv and movie_reviews_300.csv are available.")


⚠ Warning: OPENROUTER_API_KEY not found. Please check your .env file.

To set up:
1. Create an account at https://openrouter.ai/
2. Get your API key from the dashboard
3. Create a .env file in the current directory
4. Add this line: OPENROUTER_API_KEY=your_api_key_here
FEW-SHOT EXAMPLES SELECTED:

Example 1 (Positive):
  A cinematic masterpiece. The visual effects were superb and added so much depth. Do yourself a favor...

Example 2 (Negative):
  An absolute train wreck of a movie. Nothing about the opening scene worked. I wouldn't recommend thi...

Example 3 (Neutral):
  Visually it's fine, but the editing is just adequate. It was a decent way to kill an afternoon....

Total reviews to label with LLM: 150

⚠ Cannot proceed without API key. Please set up your .env file.
Creating a dummy llm_labels_150.csv for demonstration...
✓ Dummy file created. Please set up API key to get real LLM labels.


C:\Users\saikr\AppData\Local\Temp\ipykernel_29608\2165629182.py:172: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  label_samples['review_len'] = label_samples['review'].str.len()
C:\Users\saikr\AppData\Local\Temp\ipykernel_29608\2165629182.py:172: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  label_samples['review_len'] = label_samples['review'].str.len()
C:\Users\saikr\AppData\Local\Temp\ipykernel_29608\2165629182.py:172: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a Data

### Part 4.2: Noise Hunting (Cleanlab Logic)

Train a Logistic Regression model on LLM-labeled data.
Identify "High Confidence Disagreements" where the model is very confident (>0.80) but disagrees with the LLM label.

In [28]:
def find_label_errors(llm_labels, model_probs, review_texts, threshold=0.90):
    """
    Detects high-confidence disagreements between model predictions and LLM labels.
    This implements Cleanlab logic: find cases where model is confident but disagrees with LLM.
    
    Args:
        llm_labels: List/array of labels from Gemini (numeric: 0=Negative, 1=Positive, 2=Neutral)
        model_probs: Probability matrix from Logistic Regression (shape: N_samples, N_classes)
        review_texts: List of review texts (for display)
        threshold: Confidence threshold (default 0.90)
    
    Returns:
        list: List of dictionaries with suspicious review information
              Each dict contains: 'index', 'text', 'llm_label', 'model_pred', 'confidence'
    """
    # Get model predictions from probabilities (argmax)
    model_preds = np.argmax(model_probs, axis=1)
    
    # Get model confidence (max probability) for each sample
    model_confidences = np.max(model_probs, axis=1)
    
    # Convert llm_labels to numeric if they are strings
    # Map 'Positive'->1, 'Negative'->0, 'Neutral'->2
    label_mapping = {
        'Positive': 1, 'positive': 1, 'POSITIVE': 1,
        'Negative': 0, 'negative': 0, 'NEGATIVE': 0,
        'Neutral': 2, 'neutral': 2, 'NEUTRAL': 2
    }
    
    if isinstance(llm_labels[0], str):
        llm_labels_numeric = np.array([label_mapping.get(label, 2) for label in llm_labels])
    else:
        llm_labels_numeric = np.array(llm_labels)
    
    # Find disagreements where model is highly confident (> threshold) but disagrees with LLM
    # Condition: model_confidence > threshold AND model_pred != llm_label
    disagreement_mask = (model_preds != llm_labels_numeric) & (model_confidences > threshold)
    
    # Get indices of suspicious samples
    suspicious_indices = np.where(disagreement_mask)[0]
    
    # Create list of suspicious reviews with all relevant information
    suspicious_reviews = []
    
    # Label names for display
    label_names = {0: 'Negative', 1: 'Positive', 2: 'Neutral'}
    
    for idx in suspicious_indices:
        suspicious_reviews.append({
            'index': idx,
            'text': review_texts[idx],
            'llm_label': label_names[llm_labels_numeric[idx]],
            'model_pred': label_names[model_preds[idx]],
            'confidence': model_confidences[idx],
            'llm_label_numeric': llm_labels_numeric[idx],
            'model_pred_numeric': model_preds[idx]
        })
    
    # Sort by confidence (highest first) to find most egregious errors
    suspicious_reviews.sort(key=lambda x: x['confidence'], reverse=True)
    
    return suspicious_reviews


# Load LLM labels in dataframe
try:
    df_llm = pd.read_csv('llm_labels_150.csv')
    
    print("=" * 80)
    print("NOISE DETECTION USING CLEANLAB LOGIC:")
    print("=" * 80)
    print(f"Total LLM-labeled reviews: {len(df_llm)}")
    print(f"\nLLM label distribution:")
    print(df_llm['label'].value_counts())
    
    # Vectorize LLM-labeled reviews (use same vectorizer from Task 3)
    try:
        X_llm = vectorizer.transform(df_llm['review']).toarray()
        print(f"\n✓ Reviews vectorized: shape {X_llm.shape}")
    except NameError:
        print("\n⚠ Vectorizer not found. Using new TfidfVectorizer...")
        from sklearn.feature_extraction.text import TfidfVectorizer
        vectorizer_new = TfidfVectorizer(stop_words='english', max_features=1000)
        X_llm = vectorizer_new.fit_transform(df_llm['review']).toarray()
        print(f"✓ Reviews vectorized: shape {X_llm.shape}")
    
    # Convert LLM labels to numeric
    label_mapping = {
        'Positive': 1, 'positive': 1, 'POSITIVE': 1,
        'Negative': 0, 'negative': 0, 'NEGATIVE': 0,
        'Neutral': 2, 'neutral': 2, 'NEUTRAL': 2
    }
    y_llm = df_llm['label'].map(label_mapping).values
    
    # Train Logistic Regression on LLM-labeled data
    # Use same model configuration as Task 3 for consistency
    print("\n" + "=" * 80)
    print("TRAINING MODEL ON LLM LABELS:")
    print("=" * 80)
    
    model_llm = LogisticRegression(multi_class='multinomial', solver='lbfgs', max_iter=1000, random_state=42)
    model_llm.fit(X_llm, y_llm)
    
    # Evaluate model on training data (self-check)
    train_accuracy = model_llm.score(X_llm, y_llm)
    print(f"Training accuracy: {train_accuracy:.4f}")
    
    # Get probabilities on the same data (self-check)
    # Shape should be (N_samples, N_classes)
    model_probs = model_llm.predict_proba(X_llm)
    print(f"✓ Probability matrix shape: {model_probs.shape}")
    
    # Find label errors using Cleanlab logic
    print("\n" + "=" * 80)
    print("IDENTIFYING HIGH-CONFIDENCE DISAGREEMENTS:")
    print("=" * 80)
    
    suspicious = find_label_errors(
        llm_labels=df_llm['label'].values,
        model_probs=model_probs,
        review_texts=df_llm['review'].values,
        threshold=0.90  # High confidence threshold
    )
    
    print(f"\nTotal suspicious samples found: {len(suspicious)}")
    print(f"(Model confidence > 0.90 but disagrees with LLM label)")
    
    # Print top 5 suspicious reviews (or all if fewer than 5)
    print("\n" + "=" * 80)
    print("TOP 5 SUSPICIOUS REVIEWS (Potential LLM Hallucinations):")
    print("=" * 80)
    
    num_to_show = min(5, len(suspicious))
    
    if num_to_show == 0:
        print("\n✓ No high-confidence disagreements found!")
        print("  This suggests the LLM labels are consistent with the learned patterns.")
    else:
        for i in range(num_to_show):
            sample = suspicious[i]
            print(f"\n{i+1}. SUSPICIOUS SAMPLE (Index: {sample['index']})")
            print("-" * 80)
            print(f"Review: {sample['text'][:200]}...")
            print(f"\nLLM Label:       {sample['llm_label']}")
            print(f"Model Prediction: {sample['model_pred']}")
            print(f"Model Confidence: {sample['confidence']:.4f}")
            print(f"\n⚠ Analysis: Model is {sample['confidence']:.1%} confident it should be '{sample['model_pred']}',")
            print(f"            but LLM labeled it as '{sample['llm_label']}'.")
            print(f"            This is a potential labeling error or LLM hallucination.")
    
    # Additional analysis: Check if certain label pairs have more disagreements
    if len(suspicious) > 0:
        print("\n" + "=" * 80)
        print("DISAGREEMENT PATTERN ANALYSIS:")
        print("=" * 80)
        
        from collections import Counter
        disagreement_pairs = [(s['llm_label'], s['model_pred']) for s in suspicious]
        pair_counts = Counter(disagreement_pairs)
        
        print("\nMost common disagreement patterns:")
        for (llm_label, model_pred), count in pair_counts.most_common(3):
            print(f"  LLM: {llm_label:<10} vs Model: {model_pred:<10} -> {count} cases")
    
    print("\n" + "=" * 80)
    
except FileNotFoundError as e:
    print(f"⚠ Error: File not found: {e}")
    print("Please run Task 4.1 to create llm_labels_150.csv first.")


NOISE DETECTION USING CLEANLAB LOGIC:
Total LLM-labeled reviews: 150

LLM label distribution:
label
Neutral    150
Name: count, dtype: int64

⚠ Vectorizer not found. Using new TfidfVectorizer...
✓ Reviews vectorized: shape (150, 278)

TRAINING MODEL ON LLM LABELS:


c:\Users\saikr\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


ValueError: This solver needs samples of at least 2 classes in the data, but the data contains only one class: np.int64(2)

## Deliverables

**Submission Checklist:**
- [ ] Completed Jupyter Notebook with all tasks (Tasks 1-4)
- [ ] Include your label-studio annotation interface screenshot.
- [ ] gold_standard_100.csv
- [ ] weak_labels_200.csv
- [ ] llm_labels_150.json